In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from collections import Counter

import pymupdf
import requests
from bs4 import BeautifulSoup
from mitreattack.stix20 import MitreAttackData
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

import chromadb

import ollama

import asyncio
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import Faithfulness, ContextUtilization, AnswerRelevancy

from tqdm.notebook import tqdm

In [2]:
!ollama pull qwen3:8b
!ollama pull nomic-embed-text

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling a3de86cd1c13: 100% ▕██████████████████▏ 5.2 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 05a61d37b084: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df

In [3]:
#TODO - Fix invalid schema on enterprise-attacks.json.

os.makedirs("docs", exist_ok=True)

downloads = {
    "enterprise_attack.json": (
        "curl",
        "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json",
    ),
    "nist_sp_800-115.pdf": (
        "curl",
        "https://nvlpubs.nist.gov/nistpubs/Legacy/SP/nistspecialpublication800-115.pdf",
    ),
    "nist_sp_800-61r3.pdf": (
        "curl",
        "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-61r3.pdf",
    ),
    "nist_sp_800-53r5.pdf": (
        "curl",
        "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-53r5.pdf",
    ),
    "nist_csf_2.pdf": (
        "curl",
        "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
    ),
    "cert_vuln_disclosure.pdf": (
        "curl",
        "https://www.sei.cmu.edu/documents/1945/2017_003_001_503340.pdf",
    ),
    "tiber-eu.pdf": (
        "curl",
        "https://www.ecb.europa.eu/pub/pdf/other/ecb.tiber_eu_framework_2025~b32eff9a10.en.pdf?0309990e5e167a47ca4748370a949064",
    ),
    "cbest_tila.pdf": (
        "curl",
        "https://www.bankofengland.co.uk/-/media/boe/files/financial-stability/financial-sector-continuity/cbest-implementation-guide.pdf",
    ),
    "cisa_AA24-326A.pdf": (
        "wget",
        "https://www.cisa.gov/sites/default/files/2024-11/aa24-326a-enhancing-cyber-resilience-insights-from-cisa-red-team-assessment.pdf",
    ),
    "cisa_aa24-193a_silentshield.pdf": (
        "wget",
        "https://www.cisa.gov/sites/default/files/2024-07/csa-cisa-red-team%27s-operations-against-a-fceb-organization-highlights-the-necessity-of-defense-in-depth_0.pdf",
    ),
    "cisa_aa23-059a.pdf": (
        "wget",
        "https://www.cisa.gov/sites/default/files/2023-03/aa23-059a-cisa_red_team_shares_key_findings_to_improve_monitoring_and_hardening_of_networks.pdf",
    ),
}

headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"}

for fname, url in downloads.items():
    path = f"docs/{fname}"
    response = requests.get(url, headers=headers, allow_redirects=True)
    with open(path, "wb") as f:
        f.write(response.content)
    print(f"{fname} downloaded")
print("\nAll files downloaded")

InvalidSchema: No connection adapters were found for "('curl', 'https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json')"

In [4]:
attacks = MitreAttackData("docs/enterprise_attack.json")

# Build lookup table
technique_to_mitigations = attacks.get_all_mitigations_mitigating_all_techniques()
technique_to_software = attacks.get_all_software_using_all_techniques()

techniques = attacks.get_techniques(
    include_subtechniques=True,
    remove_revoked_deprecated=True,
)

print(f"Total techniques: {len(techniques)}")

Total techniques: 691


In [5]:
# build ATT&CK chunks
attck_rows = []

for technique in tqdm(techniques):
    mitre_id = attacks.get_attack_id(technique.id) or ""
    tactics = [t.name for t in attacks.get_tactics_by_technique(technique.id)]

    software_entries = technique_to_software.get(technique.id, [])
    mitigation_entries = technique_to_mitigations.get(technique.id, [])

    tools = [e["object"].name for e in software_entries if e["object"].type == "tool"]
    malware = [
        e["object"].name for e in software_entries if e["object"].type == "malware"
    ]
    mitigation_names = [e["object"].name for e in mitigation_entries]
    is_subtechnique = "." in mitre_id

    chunk_text = f"""MITRE ATT&CK Technique: {technique.name}
ID: {mitre_id}
Type: {"Sub-technique" if is_subtechnique else "Technique"}
Tactics: {", ".join(tactics)}

Description:
{getattr(technique, "description", "Not available").strip()}

Detection:
{getattr(technique, "x_mitre_detection", "Not documented").strip()}

Associated Tools: {", ".join(tools) if tools else "None documented"}
Associated Malware: {", ".join(malware) if malware else "None documented"}
Mitigations: {", ".join(mitigation_names) if mitigation_names else "None documented"}""".strip()

    attck_rows.append(
        {
            "text": chunk_text,
            "source": "MITRE ATT&CK",
            "id": mitre_id,
            "name": technique.name,
            "tactics": tactics,
            "is_subtechnique": is_subtechnique,
            "tools": tools,
            "malware": malware,
            "mitigations": mitigation_names,
        }
    )

attck_df = pd.DataFrame(attck_rows)

  0%|          | 0/691 [00:00<?, ?it/s]

In [6]:
attck_df.sample(10)

,text,source,id,name,tactics,is_subtechnique,tools,malware,mitigations
409,MITRE ATT&CK Technique: Exfiltration Over C2 C...,MITRE ATT&CK,T1041,Exfiltration Over C2 Channel,[Exfiltration],False,"[ShimRatReporter, Empire, Pupy, SILENTTRINITY,...","[OilBooster, MechaFlounder, PoetRAT, Shark, Sa...","[Network Intrusion Prevention, Data Loss Preve..."
252,MITRE ATT&CK Technique: Disable or Modify Linu...,MITRE ATT&CK,T1562.012,Disable or Modify Linux Audit System,[Defense Evasion],True,[],[Ebury],"[Audit, User Account Management]"
563,MITRE ATT&CK Technique: Launch Agent\nID: T154...,MITRE ATT&CK,T1543.001,Launch Agent,"[Persistence, Privilege Escalation]",True,[],"[Calisto, Proton, MacSpy, CrossRAT, Dok, Dacls...",[Restrict File and Directory Permissions]
202,MITRE ATT&CK Technique: Port Monitors\nID: T15...,MITRE ATT&CK,T1547.010,Port Monitors,"[Persistence, Privilege Escalation]",True,[],[],[]
394,MITRE ATT&CK Technique: Internal Defacement\nI...,MITRE ATT&CK,T1491.001,Internal Defacement,[Impact],True,[],"[ShrinkLocker, Black Basta, Diavol, ROADSWEEP,...",[Data Backup]
247,MITRE ATT&CK Technique: Hybrid Identity\nID: T...,MITRE ATT&CK,T1556.007,Hybrid Identity,"[Credential Access, Persistence, Defense Evasion]",True,[AADInternals],[],"[Multi-factor Authentication, Audit, Privilege..."
463,MITRE ATT&CK Technique: Lua\nID: T1059.011\nTy...,MITRE ATT&CK,T1059.011,Lua,[Execution],True,[],"[EvilBunny, Remsec, RedLine Stealer, Line Runn...","[Limit Software Installation, Audit, Execution..."
571,MITRE ATT&CK Technique: Clear Persistence\nID:...,MITRE ATT&CK,T1070.009,Clear Persistence,[Defense Evasion],True,[MCMD],"[SUNBURST, Bazar, IPsec Helper, PlugX, Pillowm...","[Restrict File and Directory Permissions, Remo..."
527,MITRE ATT&CK Technique: Web Session Cookie\nID...,MITRE ATT&CK,T1550.004,Web Session Cookie,"[Lateral Movement, Defense Evasion]",True,[],[],[Software Configuration]
631,MITRE ATT&CK Technique: Logon Script (Windows)...,MITRE ATT&CK,T1037.001,Logon Script (Windows),"[Persistence, Privilege Escalation]",True,[],"[Attor, JHUHUGIT, KGH_SPY, Zebrocy]",[Restrict Registry Permissions]


In [7]:
PDF_SOURCES = {
    "docs/cisa_AA24-326A.pdf": "CISA Red Team Assessment AA24-326A",
    "docs/cisa_aa24-193a_silentshield.pdf": "CISA Red Team Assessment AA24-193A SILENTSHIELD",
    "docs/cisa_aa23-059a.pdf": "CISA Red Team Assessment AA23-059A",
    "docs/nist_sp_800-115.pdf": "NIST SP 800-115 Information Security Testing",
    "docs/nist_sp_800-61r3.pdf": "NIST SP 800-61r3 Incident Response",
    "docs/nist_sp_800-53r5.pdf": "NIST SP 800-53r5 Security and Privacy Controls",
    "docs/nist_csf_2.pdf": "NIST Cybersecurity Framework 2.0",
    "docs/cert_vuln_disclosure.pdf": "CERT Guide to Coordinated Vulnerability Disclosure",
    "docs/tiber-eu.pdf": "EU Threat Intelligence-Based Ethical Red Teaming",
    "docs/cbest_tila.pdf": "CBEST Threat Intelligence-Led Assessments",
}

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95,
)

rows = []

for file_path, doc_name in PDF_SOURCES.items():
    tqdm.write(f"Processing {doc_name}")
    try:
        doc = pymupdf.open(file_path)

        for i, page in enumerate(tqdm(doc, desc=f"Pages in {doc_name}")):
            page_text = page.get_text().strip()
            if not page_text:
                continue

            chunks = splitter.create_documents([page_text])
            
            for chunk in chunks:
                rows.append({
                    "text": chunk.page_content,
                    "source": doc_name,
                    "page": i+1,
                })
        
        doc.close()
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

doc_df = pd.DataFrame(rows)

if not doc_df.empty:
    doc_df = doc_df[doc_df["text"].str.len() > 50].reset_index(drop=True)
else:
    print("No text chunks were extracted.")

Processing CISA Red Team Assessment AA24-326A


Pages in CISA Red Team Assessment AA24-326A:   0%|          | 0/36 [00:00<?, ?it/s]

Processing CISA Red Team Assessment AA24-193A SILENTSHIELD


Pages in CISA Red Team Assessment AA24-193A SILENTSHIELD:   0%|          | 0/29 [00:00<?, ?it/s]

Processing CISA Red Team Assessment AA23-059A


...

Pages in NIST SP 800-115 Information Security Testing:   0%|          | 0/80 [00:00<?, ?it/s]

MuPDF error: format error: No default Layer config

Error processing docs/nist_sp_800-115.pdf: the input length exceeds the context length (status code: 400)
Processing NIST SP 800-61r3 Incident Response


Pages in NIST SP 800-61r3 Incident Response:   0%|          | 0/48 [00:00<?, ?it/s]

Processing NIST SP 800-53r5 Security and Privacy Controls


Pages in NIST SP 800-53r5 Security and Privacy Controls:   0%|          | 0/492 [00:00<?, ?it/s]

In [ ]:
print(doc_df.sample(5))

In [ ]:
# --- PTES: Scrape and chunk all seven sections ---
PTES_SECTIONS = {
    "Pre-engagement Interactions": "http://www.pentest-standard.org/index.php/Pre-engagement",
    "Intelligence Gathering": "http://www.pentest-standard.org/index.php/Intelligence_Gathering",
    "Threat Modeling": "http://www.pentest-standard.org/index.php/Threat_Modeling",
    "Vulnerability Analysis": "http://www.pentest-standard.org/index.php/Vulnerability_Analysis",
    "Exploitation": "http://www.pentest-standard.org/index.php/Exploitation",
    "Post Exploitation": "http://www.pentest-standard.org/index.php/Post_Exploitation",
    "Reporting": "http://www.pentest-standard.org/index.php/Reporting",
}


def scrape_ptes_section(url, section_name):
    """Scrape a PTES MediaWiki section and return clean text."""
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"  Failed to fetch {section_name}: {e}")
        return None

    soup = BeautifulSoup(r.text, "html.parser")
    content_div = soup.find("div", {"id": "mw-content-text"}) or soup.find("body")
    if not content_div:
        return None

    for tag in content_div.find_all(
        ["script", "style", "table", "sup", "span.editsection"]
    ):
        tag.decompose()

    lines = [l.strip() for l in content_div.get_text(separator="\n").splitlines()]
    clean_text = "\n\n".join(l for l in lines if l)
    return clean_text


def chunk_ptes_section(text, section_name, chunk_size=800, overlap=150):
    """Split a PTES section into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    return [
        {
            "text": s.strip(),
            "metadata": {"source": "PTES", "section": section_name, "chunk_index": i},
        }
        for i, s in enumerate(splitter.split_text(text))
        if len(s.strip()) >= 50
    ]


ptes_chunks = []
for section_name, url in PTES_SECTIONS.items():
    print(f"Scraping: {section_name}...")
    text = scrape_ptes_section(url, section_name)
    if text:
        chunks = chunk_ptes_section(text, section_name)
        ptes_chunks.extend(chunks)
        print(f"  → {len(chunks)} chunks")

print(f"\nTotal PTES chunks: {len(ptes_chunks)}")

In [ ]:
# --- Combine all sources ---
all_chunks = attck_chunks + pdf_chunks + ptes_chunks

print("Chunks per source:")
for source, count in Counter(c["metadata"]["source"] for c in all_chunks).items():
    print(f"  {source}: {count}")
print(f"  TOTAL: {len(all_chunks)}")

## Embedding

In [ ]:
# --- Load BGE-M3 embedding model ---
embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("BGE-M3 loaded")

In [ ]:
def truncate_chunk(text, max_chars=3000):
    """Safety truncation — BGE-M3 max is 8192 tokens but very long inputs slow inference."""
    return text[:max_chars] if len(text) > max_chars else text


all_texts = [truncate_chunk(c["text"]) for c in all_chunks]
all_dense = []
all_sparse = []
BATCH_SIZE = 16

for start in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Embedding chunks"):
    batch = all_texts[start : start + BATCH_SIZE]

    encoded = embedder.encode(
        batch, return_dense=True, return_sparse=True, batch_size=BATCH_SIZE
    )

    all_dense.append(encoded["dense_vecs"])
    all_sparse.extend(encoded["lexical_weights"])

dense_vecs = np.vstack(all_dense)
sparse_weights = all_sparse

print(f"Dense shape:    {dense_vecs.shape}")
print(f"Sparse entries: {len(sparse_weights)}")

In [ ]:
# --- Store dense vectors in ChromaDB ---
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete and recreate if rebuilding from scratch
try:
    chroma_client.delete_collection("redteam_rag")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="redteam_rag", metadata={"hnsw:space": "cosine"}
)

ids = [str(i) for i in range(len(all_chunks))]
documents = [c["text"] for c in all_chunks]

# ChromaDB requires flat metadata (no nested lists)
metadatas = []
for c in all_chunks:
    meta = dict(c["metadata"])
    for k, v in meta.items():
        if isinstance(v, list):
            meta[k] = ", ".join(v) if v else ""
    metadatas.append(meta)

BATCH_SIZE = 500
for i in range(0, len(all_chunks), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(all_chunks))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        embeddings=dense_vecs[i:end].tolist(),
        metadatas=metadatas[i:end],
    )
    print(f"  Stored batch {i}:{end}")

print(f"Total stored in ChromaDB: {collection.count()}")

In [ ]:
# --- Save sparse weights to disk ---
sparse_store = {str(i): sparse_weights[i] for i in range(len(all_chunks))}

with open("sparse_weights.pkl", "wb") as f:
    pickle.dump(sparse_store, f)

print("Sparse weights saved")

# Verify consistency
assert (
    collection.count() == len(sparse_store) == len(all_chunks)
), "Mismatch between stores — recheck ingestion"
print("All stores consistent")

In [ ]:
# --- Reload from disk (run this cell after a kernel restart) ---
embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection("redteam_rag")

with open("sparse_weights.pkl", "rb") as f:
    sparse_store = pickle.load(f)

print(f"Collection: {collection.count()} chunks")
print(f"Sparse:     {len(sparse_store)} entries")

## Retrieval

In [ ]:
# --- Hybrid retrieval functions ---


def dense_query(query_vec, collection, top_k=10):
    """Retrieve top-k chunks by cosine similarity."""
    results = collection.query(
        query_embeddings=query_vec.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    ids = [int(i) for i in results["ids"][0]]
    scores = [1 - d for d in results["distances"][0]]
    return ids, scores


def sparse_query(query_sparse, sparse_store, top_k=10):
    """Retrieve top-k chunks by lexical weight dot product."""
    scores = {}
    for idx, doc_sparse in sparse_store.items():
        shared = set(query_sparse.keys()) & set(doc_sparse.keys())
        scores[int(idx)] = sum(query_sparse[t] * doc_sparse[t] for t in shared)
    sorted_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return sorted_ids, [scores[i] for i in sorted_ids]


def reciprocal_rank_fusion(ranked_lists, k=60):
    """Combine ranked lists via RRF (Cormack et al., 2009). k=60 is the standard constant."""
    rrf_scores = {}
    for ranked_ids in ranked_lists:
        for rank, doc_id in enumerate(ranked_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return rrf_scores


def hybrid_retrieve(query, embedder, collection, sparse_store, all_chunks, top_k=10):
    """Encode query, run dense + sparse retrieval, fuse with RRF."""
    q_encoded = embedder.encode([query], return_dense=True, return_sparse=True)
    q_dense = q_encoded["dense_vecs"]
    q_sparse = q_encoded["lexical_weights"][0]

    dense_ids, _ = dense_query(q_dense, collection, top_k)
    sparse_ids, _ = sparse_query(q_sparse, sparse_store, top_k)

    rrf_scores = reciprocal_rank_fusion([dense_ids, sparse_ids])
    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

    return [all_chunks[i] for i in sorted_ids], sorted_ids

In [ ]:
# --- Reranker (cross-encoder, bge-reranker-v2-m3) ---
# Loaded once; freed after use to reclaim VRAM before generation
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)


def rerank(query, candidates, top_k=3):
    """Score query-candidate pairs, return top-k by reranker score."""
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked[:top_k]], [s for s, _ in ranked[:top_k]]

## Generation

In [ ]:
# --- System prompt and generation function ---
SYSTEM_PROMPT = """You are an expert red team cybersecurity advisor.
Your role is to provide accurate, actionable guidance to red team practitioners.
Answer ONLY based on the provided context.
If the context does not contain enough information, say so explicitly.
Always cite the source document (e.g. MITRE ATT&CK T1550.002, PTES, CISA)
for each piece of advice you give."""


def generate(query, context_chunks, stream=True):
    """Generate a response from Qwen3:8b using the retrieved context."""
    context = "\n\n---\n\n".join(
        [
            f"[Source: {c['metadata']['source']} | "
            f"{c['metadata'].get('id', c['metadata'].get('section', ''))}]\n"
            f"{c['text']}"
            for c in context_chunks
        ]
    )

    response_stream = ollama.chat(
        model="qwen3:8b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
        options={"temperature": 0.1, "think": False},
        stream=stream,
    )

    if stream:
        full_response = ""
        for chunk in response_stream:
            token = chunk["message"]["content"]
            print(token, end="", flush=True)
            full_response += token
        print()
        return full_response
    else:
        return response_stream["message"]["content"]

## RAG

Measures:
* [Faithfulness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/)

$\text{Faithfulness Score} \in (0, 1) = \frac{\text{Number of claims in the response supported by the retrieved context}}{\text{Total number of claims in the response}}$

* [Context Utilisation](https://docs.ragas.io/en/v0.1.21/concepts/metrics/context_utilization.html)

$ x \in (0, 1)$

* [Answer Relevance](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/)

$ x \in (-1, 1)$

In [ ]:
# --- Full RAG pipeline ---
def rag_pipeline(query, top_k_retrieve=10, top_k_rerank=3, verbose=True):
    """
    End-to-end pipeline:
    1. Hybrid retrieval (dense + sparse + RRF) → top_k_retrieve candidates
    2. Cross-encoder reranking → top_k_rerank final chunks
    3. LLM generation with retrieved context
    """
    if verbose:
        print(f"Query: {query}\n")

    # Stage 1 — Hybrid retrieval
    candidates, candidate_ids = hybrid_retrieve(
        query, embedder, collection, sparse_store, all_chunks, top_k=top_k_retrieve
    )
    if verbose:
        print(f"Retrieved {len(candidates)} candidates")

    # Stage 2 — Rerank
    final_chunks, rerank_scores = rerank(query, candidates, top_k=top_k_rerank)
    if verbose:
        print(f"Reranked to top {top_k_rerank}:")
        for i, (chunk, score) in enumerate(zip(final_chunks, rerank_scores)):
            src = chunk["metadata"]["source"]
            name = chunk["metadata"].get("id", chunk["metadata"].get("section", ""))
            print(f"  {i+1}. [{score:.3f}] {src} | {name}")

    # Stage 3 — Generate
    answer = generate(query, final_chunks, stream=verbose)
    if verbose:
        print(f"\nAnswer:\n{answer}")

    return {
        "query": query,
        "retrieved": candidates,
        "final_chunks": final_chunks,
        "rerank_scores": rerank_scores,
        "answer": answer,
    }


# Smoke test
_ = rag_pipeline(
    "What reconnaissance techniques should I use before an assumed breach engagement?"
)

## RAGAS Evaluation

In [ ]:
# --- Configure RAGAS with local Ollama ---
ragas_client = AsyncOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

ragas_llm = llm_factory(
    "qwen3:8b",
    client=ragas_client,
    max_tokens=4096,
    extra_body={"options": {"think": False, "num_ctx": 8192}},
)
ragas_embeddings = embedding_factory(
    "openai", model="nomic-embed-text", client=ragas_client
)


faithfulness_scorer = Faithfulness(llm=ragas_llm)
context_utilization_scorer = ContextUtilization(llm=ragas_llm)
answer_relevancy_scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

print("RAGAS scorers initialised")

In [ ]:
# --- Run all 10 practitioner queries through the full pipeline ---
QUERIES = [
    "What reconnaissance techniques should I use before an assumed breach engagement?",
    "How do I establish persistence on a Windows host without triggering EDR?",
    "What are the rules of engagement considerations for a purple team exercise?",
    "How should lateral movement be documented during a red team engagement?",
    "What MITRE ATT&CK techniques are most relevant to Active Directory attacks?",
    "How do I scope a red team engagement for a mid-sized financial organisation?",
    "What is the difference between a red team and a penetration test?",
    "How should I handle discovering a critical zero-day during a client engagement?",
    "What tools are commonly used for command and control in adversary simulation?",
    "How do I write an executive summary for a red team report?",
]

results = []
for query in tqdm(QUERIES):
    result = rag_pipeline(query)
    results.append(result)

print(f"\nAll {len(results)} queries complete")

In [ ]:
# Fix max_token limit issue

MAX_CTX_CHARS = 1500
MAX_ANSWER_CHARS = 800


# --- Compute RAGAS metrics for all results ---
async def evaluate_single(query, answer, contexts):
    # Truncate each context chunk before scoring
    contexts = [c[:MAX_CTX_CHARS] for c in contexts]
    answer = answer[:MAX_ANSWER_CHARS]
    faith = await faithfulness_scorer.ascore(
        user_input=query, response=answer, retrieved_contexts=contexts
    )
    ctx_util = await context_utilization_scorer.ascore(
        user_input=query, response=answer, retrieved_contexts=contexts
    )
    relev = await answer_relevancy_scorer.ascore(user_input=query, response=answer)
    return {
        "query": query,
        "faithfulness": round(faith.value, 3),
        "context_utilization": round(ctx_util.value, 3),
        "answer_relevancy": round(relev.value, 3),
    }


async def evaluate_all(results):
    records = []
    for r in tqdm(results):
        contexts = [c["text"] for c in r["final_chunks"]]
        scores = await evaluate_single(r["query"], r["answer"], contexts)
        records.append(scores)
        print(
            f"F={scores['faithfulness']:.3f} | "
            f"CU={scores['context_utilization']:.3f} | "
            f"AR={scores['answer_relevancy']:.3f}"
        )
    return records


# Jupyter supports top-level await
eval_records = await evaluate_all(results)

In [ ]:
# --- Results table ---
METRIC_COLS = ["faithfulness", "context_utilization", "answer_relevancy"]

df = pd.DataFrame(eval_records)
df.index = [f"Q{i+1}" for i in range(len(df))]

print("=== RAGAS Evaluation Results ===")
print(df[["query"] + METRIC_COLS].to_string())

print("\n=== Mean Scores ===")
print(df[METRIC_COLS].mean().round(3).to_string())